# Day 3: Baseline ML with Enhanced DIP Features
Trains RF and SVM using the standardized `src.features` module.

In [1]:
import os
import sys
import zipfile
import urllib.request
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DATA_DIR = Path('/content/data/processed')
    FIGURES_DIR = Path('/content/report/figures')
else:
    DATA_DIR = Path('../data/processed')
    FIGURES_DIR = Path('../report/figures')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# If running in Colab and data not present, download EuroSAT
if IN_COLAB and not DATA_DIR.exists():
    import shutil
    print('Downloading EuroSAT RGB dataset (~90 MB)...')
    url = 'https://madm.dfki.de/files/sentinel/EuroSAT.zip'
    urllib.request.urlretrieve(url, '/content/EuroSAT.zip')
    print('Extracting...')
    with zipfile.ZipFile('/content/EuroSAT.zip', 'r') as z:
        z.extractall('/content/EuroSAT_raw')
    raw_root = Path('/content/EuroSAT_raw/2750')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for cls_dir in raw_root.iterdir():
        if cls_dir.is_dir():
            shutil.copytree(cls_dir, DATA_DIR / cls_dir.name)
    print('Done! Classes found:', sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()]))
elif not DATA_DIR.exists():
    raise FileNotFoundError('Data directory not found: ' + str(DATA_DIR))
else:
    print('Using DATA_DIR:', DATA_DIR)
    print('Classes found:', sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()]))


Using DATA_DIR: /content/data/processed
Classes found: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [2]:
import cv2
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from pathlib import Path
from tqdm import tqdm
import sys
import os
import json

# Try to locate project root that contains src/features.py in local or Colab setups.
def _find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('..').resolve(),
        Path('/content/dip_project'),
        Path('/content/drive/MyDrive/dip_project'),
    ]
    for root in candidates:
        if (root / 'src' / 'features.py').exists() and (root / 'src' / 'utils.py').exists():
            return root
    return None

project_root = _find_project_root()
if project_root is not None:
    sys.path.insert(0, str(project_root))

try:
    from src.features import extract_lulc_features
    from src.utils import update_metrics
except ModuleNotFoundError:
    print('src package not found; using notebook-local fallback helpers.')

    def extract_lulc_features(img_rgb):
        """Fallback feature extractor: color stats + simple texture cues."""
        img = img_rgb.astype(np.float32)

        means = img.mean(axis=(0, 1))
        stds = img.std(axis=(0, 1))

        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        gray_mean = float(gray.mean())
        gray_std = float(gray.std())

        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        grad_mag = np.sqrt(gx**2 + gy**2)
        grad_mean = float(grad_mag.mean())
        grad_std = float(grad_mag.std())

        return np.concatenate([
            means,
            stds,
            np.array([gray_mean, gray_std, grad_mean, grad_std], dtype=np.float32),
        ])

    def update_metrics(model_name, acc):
        """Fallback metric logger for standalone notebook runs."""
        metrics_path = Path('metrics_fallback.json')
        records = []
        if metrics_path.exists():
            with open(metrics_path, 'r', encoding='utf-8') as f:
                records = json.load(f)
        records.append({'model': model_name, 'accuracy': acc})
        with open(metrics_path, 'w', encoding='utf-8') as f:
            json.dump(records, f, indent=2)
        print(f'Logged {model_name} with {acc}% accuracy to {metrics_path}.')

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)


src package not found; using notebook-local fallback helpers.


In [3]:
data = []
labels = []
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])

print('Generating Standardized Dataset...')
for i, cls in enumerate(classes):
    paths = list((DATA_DIR / cls).glob('*.jpg'))[:500] # Increased for better accuracy
    for img_path in tqdm(paths, desc=f'Class {cls}'):
        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            feat = extract_lulc_features(img_rgb)
            data.append(feat)
            labels.append(i)

X = np.array(data)
y = np.array(labels)
print(f'Feature vector size: {X.shape[1]}')

Generating Standardized Dataset...


Class AnnualCrop:  31%|███       | 154/500 [00:00<00:00, 1535.86it/s]

Class SeaLake: 100%|██████████| 500/500 [00:00<00:00, 1742.24it/s]

Feature vector size: 10


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
joblib.dump(rf, MODEL_DIR / 'rf_baseline.joblib')
acc_rf = accuracy_score(y_test, rf.predict(X_test))
update_metrics('DIP Features + RF', round(acc_rf * 100, 2))

print('Training SVM...')
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test))
update_metrics('SVM', round(acc_svm * 100, 2))

print('Training KNN (Advanced Comparison)...')
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test))
update_metrics('K-Nearest Neighbors', round(acc_knn * 100, 2))

print('Training Gradient Boosting (XGBoost alternative)...')
gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbc.fit(X_train, y_train)
acc_gbc = accuracy_score(y_test, gbc.predict(X_test))
update_metrics('Gradient Boosting (XGB)', round(acc_gbc * 100, 2))

print('\nReport (RF Baseline):')
print(classification_report(y_test, rf.predict(X_test), target_names=classes))

Training Random Forest...
Logged DIP Features + RF with 77.0% accuracy to metrics_fallback.json.
Training SVM...
Logged SVM with 72.8% accuracy to metrics_fallback.json.
Training KNN (Advanced Comparison)...
Logged K-Nearest Neighbors with 73.5% accuracy to metrics_fallback.json.
Training Gradient Boosting (XGBoost alternative)...
Logged Gradient Boosting (XGB) with 78.0% accuracy to metrics_fallback.json.

Report (RF Baseline):
                      precision    recall  f1-score   support

          AnnualCrop       0.79      0.81      0.80       100
              Forest       0.90      0.95      0.93       100
HerbaceousVegetation       0.77      0.61      0.68       100
             Highway       0.54      0.50      0.52       100
          Industrial       0.89      0.93      0.91       100
             Pasture       0.74      0.73      0.74       100
       PermanentCrop       0.61      0.66      0.63       100
         Residential       0.89      0.92      0.91       100
        